# AAC Dataset Annotation Pipeline

This notebook annotates the **ARASAAC CommonGen** dataset using a locally loaded instruction-tuned LLM (Mistral-7B-Instruct-v0.3).

Each row in the dataset contains a short activity sentence and a list of key concepts. The goal is to enrich each row with:
- `time_of_day`: the most natural time slot for the activity (morning / afternoon / evening / night)
- `event_time`: a concrete clock time (HH:MM) sampled within that slot
- `caregiver_clear`: a clear, specific sentence a caregiver could say, with `{TIME}` as placeholder
- `caregiver_vague`: a short, implicit fragment (no activity name, informal tone)
- `schedule`: a list of calendar events, if the activity is organised or out-of-home

## 1. Configuration

In [ ]:
from pathlib import Path

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID          = "mistralai/Mistral-7B-Instruct-v0.3"
LOAD_IN_4BIT      = True   # enables 4-bit NF4 quantisation to reduce VRAM usage
MAX_NEW_TOKENS    = 256    # max tokens the model can generate per row
MAX_PROMPT_LENGTH = 2048   # prompt is truncated if longer than this

# ── Dataset ───────────────────────────────────────────────────────────────────
HF_DATASET = "disi-unibo-nlp-students/ARASAAC_CommonGen_new_dataset"
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env for local development (no-op on cluster where HF_TOKEN is set via env)
load_dotenv(Path('../../app/.env'))
HF_TOKEN   = os.environ.get("HF_TOKEN", "")

# ── Batching ──────────────────────────────────────────────────────────────────
BATCH_SIZE = 64   # number of rows processed in parallel during generation

# ── Retry logic ───────────────────────────────────────────────────────────────
MAX_ROW_RETRIES        = 3   # how many times to retry a single failing row within a batch
MAX_ANNOTATION_RETRIES = 5   # how many full passes over the dataset to attempt

# ── Output paths ──────────────────────────────────────────────────────────────
WORK_DIR = Path(".")
WORK_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH       = WORK_DIR / "eval_raw.parquet"         # raw dataset snapshot
ANNOTATED_PATH = WORK_DIR / "eval_annotated.parquet"   # final annotated output
LOG_PATH       = WORK_DIR / "annotation_log.jsonl"     # per-row annotation log (used for resuming)

## 2. Imports & Logging

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduces CUDA OOM fragmentation

import json
import logging
import re
import time
from datetime import datetime

import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
)

logging.basicConfig(
    level    = logging.INFO,
    format   = "%(asctime)s %(levelname)s %(message)s",
    handlers = [
        logging.FileHandler(WORK_DIR / "annotate.log"),  # write to file
    ],
)
log = logging.getLogger("annotate")

device = "cuda" if torch.cuda.is_available() else "cpu"
log.info("Device: %s", device)
if device == "cuda":
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        log.info("  GPU %d: %s  (%.1f GB VRAM)", i, props.name, props.total_memory / 1e9)
else:
    log.warning("No GPU found: annotation will be very slow on CPU.")

## 3. Load / Download Dataset

If a local parquet snapshot already exists we load it directly, avoiding repeated HuggingFace downloads. All splits are concatenated into a single flat DataFrame.

In [ ]:
def load_raw_dataset() -> pd.DataFrame:
    if RAW_PATH.exists():
        log.info("Raw parquet found: loading from %s", RAW_PATH)
        df = pd.read_parquet(RAW_PATH)
        log.info("Loaded %d rows", len(df))
        return df

    log.info("Downloading '%s' from HuggingFace ...", HF_DATASET)
    hf_ds  = load_dataset(HF_DATASET, token=HF_TOKEN)
    splits = list(hf_ds.keys())
    log.info("Splits found: %s", splits)

    # Concatenate all splits into one DataFrame and reset the index
    df = pd.concat([hf_ds[s].to_pandas() for s in splits], ignore_index=True)
    log.info("Total rows: %d", len(df))

    df.to_parquet(RAW_PATH, index=False)
    log.info("Saved raw snapshot: %s", RAW_PATH)
    return df

df_raw = load_raw_dataset()
log.info("Dataset shape: %s  columns: %s", df_raw.shape, list(df_raw.columns))

## 4. Load Model & Tokenizer

In [ ]:
log.info("Loading tokenizer for %s ...", MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    # Some models don't define a pad token; reusing eos_token is the standard fix
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for correct batched generation

quant_cfg = None
if LOAD_IN_4BIT and device == "cuda":
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_compute_dtype    = torch.bfloat16,
        bnb_4bit_use_double_quant = True,   # double quantisation saves a bit more memory
        bnb_4bit_quant_type       = "nf4",  # NormalFloat4 is best for LLM weights
    )
    log.info("4-bit NF4 quantisation enabled")

log.info("Loading model weights ...")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = quant_cfg,
    device_map          = "auto" if device == "cuda" else None,
    torch_dtype         = torch.bfloat16 if device == "cuda" and not LOAD_IN_4BIT else None,
    trust_remote_code   = True,
)
model.eval()
log.info("Model loaded in %.1f s", time.time() - t0)

if device == "cuda":
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        log.info("  GPU %d VRAM after load: %.1f / %.1f GB", i, alloc, total)

## 5. Time Sampling, Semantic Check & Prompt Builder

### Design overview

The model is responsible for choosing `time_of_day` (morning / afternoon / evening / night) based on the activity. Once we know the slot, we **deterministically sample a concrete clock time** (`event_time`) within it. This separation means:
- the model focuses on *semantic* correctness (what time of day fits the activity)
- the code handles *numeric* correctness (a valid clock time within that slot)

**NB:** this is necessary as in previous experiments, when the model was asked to generate a concrete time directly, it often produced invalid times (broke the HH:MM format and the 0-23 range) and also producing often the exact same time for many activities (like 14:00 for afternoon), which is not ideal for downstream diversity.

### Semantic override

Some activities have an *unambiguous* time of day (e.g. sleep: night, breakfast: morning). For these we use a keyword lookup table (`_SEMANTIC_KEYWORDS`) to override the model's choice **after** generation. This prevents the model from placing for example "sleep" in the morning slot.

### Time slots

Internally we distinguish five fine-grained slots, but the model only sees four public labels:

| Public label | Internal slot | Hour range |
|---|---|---|
| morning | morning | 05 – 12 |
| afternoon | afternoon | 13 – 17 |
| evening | evening | 18 – 20 |
| night | night_early | 21 – 23 |
| night | night_late | 00 – 04 |

`night` maps to two internal slots so we can sample realistic times for both night events (22:00) and sleep (02:00).

**Time format:** all clock times in this pipeline use `HH:MM` format. Seconds are always `00` and are never stored or exposed; the code works exclusively in `HH:MM`.

### Prompt strategy

We use an **8-shot prompt** covering all four time slots, with two examples each. The few-shot examples are the main driver of output quality. This mix of examples allows the model to learn the concept of time slots and their associated activities in a balanced perspective, while also providing a clear format to follow for both `time_of_day` and `caregiver` fields.

In [ ]:
import random as _random

# ── Semantic keyword table ────────────────────────────────────────────────────
# Maps public time_of_day labels to lists of trigger words.
# If the activity sentence contains any of these words, the model's
# time_of_day choice is overridden with the corresponding label.
_SEMANTIC_KEYWORDS: dict[str, list[str]] = {
    "night"    : ["sleep", "bedtime", "night fill", "live concert", "live performance"],
    "morning"  : ["breakfast", "sunrise"],
    "afternoon": ["nap", "lunch"],
    "evening"  : ["dinner", "supper"],
}

# ── Internal slot definitions ─────────────────────────────────────────────────
# Each internal slot maps to: (public_label, hour_min, hour_max)
_SLOT_INFO: dict[str, tuple[str, int, int]] = {
    "morning"    : ("morning",   5,  12),
    "afternoon"  : ("afternoon", 13, 17),
    "evening"    : ("evening",   18, 20),
    "night_early": ("night",     21, 23),
    "night_late" : ("night",      0,  4),
}

# Fallback distribution used when the model produces an invalid time_of_day
_DEFAULT_SLOTS   = ["morning", "afternoon", "evening", "night_early", "night_late"]
_DEFAULT_WEIGHTS = [0.30,       0.30,        0.20,      0.12,          0.08]

# Maps public label to internal slot(s) to sample from
# "night" has two because it covers both early-night and late-night activities
_TOD_TO_INTERNAL: dict[str, list[str]] = {
    "morning"  : ["morning"],
    "afternoon": ["afternoon"],
    "evening"  : ["evening"],
    "night"    : ["night_early", "night_late"],
}


def _semantic_check(sentence: str, concepts: list) -> str | None:
    """Return the forced public time_of_day if the sentence contains a keyword, else None."""
    combined = (sentence + " " + " ".join(str(c) for c in concepts)).lower()
    for tod, words in _SEMANTIC_KEYWORDS.items():
        for w in words:
            if w in combined:
                return tod
    return None


def _sample_event_time(time_of_day: str) -> tuple[str, str, str]:
    """Sample a concrete HH:MM within the slot corresponding to time_of_day.

    Seconds are always 00; all times use the HH:MM format.

    Returns (internal_category, public_tod, event_time_string).
    If time_of_day is not a recognised label (the model hallucinated),
    we fall back to a weighted random draw over all slots.
    """
    internal_cats = _TOD_TO_INTERNAL.get(time_of_day)
    if internal_cats is None:
        # The model produced something unexpected: fall back to a weighted random slot
        cat = _random.choices(_DEFAULT_SLOTS, weights=_DEFAULT_WEIGHTS)[0]
    elif len(internal_cats) == 1:
        cat = internal_cats[0]
    else:
        # "night" has two internal slots: pick one at random
        cat = _random.choice(internal_cats)

    tod, lo, hi = _SLOT_INFO[cat]
    hour   = _random.randint(lo, hi)
    minute = _random.choice([0, 15, 30, 45])  # round minutes only, more realistic
    return cat, tod, f"{hour:02d}:{minute:02d}"


# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a JSON generator for AAC (Augmentative and Alternative "
    "Communication) activity annotation.\n"
    "Given an activity sentence and key concepts, output a single JSON object.\n"
    "Output ONLY compact single-line JSON: no newlines, no indentation, "
    "no spaces after colons or commas, no prose, no markdown fences.\n"
    "Use the literal string {TIME} as a placeholder wherever the clock time "
    "belongs: it will be replaced programmatically with HH:MM."
)

# ── Few-shot examples ─────────────────────────────────────────────────────────
# Two examples per time slot to help the model generalise across all four labels.
_FEW_SHOT_EXAMPLES = [
    # ── Morning ────────────────────────────────────────────────────────────
    {
        "sentence": "a girl swim in the pool",
        "concepts": "girl, swim, pool",
        "json"    : (
            '{"time_of_day":"morning",'
            '"caregiver_clear":"She has swimming at the pool at {TIME} this morning",'
            '"caregiver_vague":"the pool thing, same as Tuesday",'
            '"schedule":[{"title":"Swimming lesson","start_time":"{TIME}",'
            '"location":null,"description":null}]}'
        ),
    },
    {
        "sentence": "a boy ride horse at the farm",
        "concepts": "boy, ride, horse, farm",
        "json"    : (
            '{"time_of_day":"morning",'
            '"caregiver_clear":"He has horse riding at the farm at {TIME} this morning",'
            '"caregiver_vague":"that animal thing he loves",'
            '"schedule":[{"title":"Horse riding","start_time":"{TIME}",'
            '"location":null,"description":null}]}'
        ),
    },
    # ── Afternoon ──────────────────────────────────────────────────────────
    {
        "sentence": "a boy watch television in the living room",
        "concepts": "boy, watch, television, living room",
        "json"    : (
            '{"time_of_day":"afternoon",'
            '"caregiver_clear":"He is watching TV in the living room at {TIME}",'
            '"caregiver_vague":"the screen thing, same channel",'
            '"schedule":[]}'
        ),
    },
    {
        "sentence": "a child play soccer in the park",
        "concepts": "child, play, soccer, park",
        "json"    : (
            '{"time_of_day":"afternoon",'
            '"caregiver_clear":"The child has soccer at the park at {TIME} this afternoon",'
            '"caregiver_vague":"the outdoor thing, same spot",'
            '"schedule":[{"title":"Soccer practice","start_time":"{TIME}",'
            '"location":null,"description":null}]}'
        ),
    },
    # ── Evening ────────────────────────────────────────────────────────────
    {
        "sentence": "a family eat dinner at home",
        "concepts": "family, eat, dinner, home",
        "json"    : (
            '{"time_of_day":"evening",'
            '"caregiver_clear":"The family is having dinner at home at {TIME} tonight",'
            '"caregiver_vague":"that food thing, he keeps asking",'
            '"schedule":[]}'
        ),
    },
    {
        "sentence": "a woman do yoga at the gym",
        "concepts": "woman, do, yoga, gym",
        "json"    : (
            '{"time_of_day":"evening",'
            '"caregiver_clear":"She has yoga class at the gym at {TIME} this evening",'
            '"caregiver_vague":"that stretching thing, the usual place",'
            '"schedule":[{"title":"Yoga class","start_time":"{TIME}",'
            '"location":null,"description":null}]}'
        ),
    },
    # ── Night ──────────────────────────────────────────────────────────────
    {
        "sentence": "a group watch live concert",
        "concepts": "group, watch, live concert",
        "json"    : (
            '{"time_of_day":"night",'
            '"caregiver_clear":"They are at the live concert at {TIME} tonight",'
            '"caregiver_vague":"the concert thing, same venue",'
            '"schedule":[{"title":"Live concert","start_time":"{TIME}",'
            '"location":null,"description":null}]}'
        ),
    },
    {
        "sentence": "a child sleep in the bedroom",
        "concepts": "child, sleep, bedroom",
        "json"    : (
            '{"time_of_day":"night",'
            '"caregiver_clear":"The child is sleeping in the bedroom at {TIME}",'
            '"caregiver_vague":"bedtime, same routine",'
            '"schedule":[]}'
        ),
    },
]


def _build_examples_text() -> str:
    """Format few-shot examples into the input/output template used in the prompt."""
    parts = []
    for ex in _FEW_SHOT_EXAMPLES:
        parts.append(
            f'Input: sentence="{ex["sentence"]}", concepts=[{ex["concepts"]}]\n'
            f'Output: {ex["json"]}'
        )
    return "\n\n".join(parts)

_EXAMPLES_TEXT = _build_examples_text()


def _build_prompt(sentence: str, concepts: list) -> str:
    """Build the full chat-formatted prompt for a single row."""
    concepts_str = ", ".join(str(c) for c in concepts)

    user_msg = (
        "Generate a JSON annotation for an AAC (Augmentative and Alternative "
        "Communication) activity.\n\n"

        "Input fields:\n"
        "  sentence: simple English sentence describing the activity.\n"
        "  concepts: key content words from the sentence.\n\n"

        "Output fields (one compact JSON object):\n"
        "  time_of_day      The most appropriate time of day for this activity.\n"
        '                   Exactly one of: "morning", "afternoon", "evening", "night".\n'
        "  caregiver_clear  Specific natural sentence for a caregiver. Use third-person\n"
        "                   pronouns or role nouns (he/she/they/the child/the woman/etc.).\n"
        "                   Must include the literal placeholder {TIME} where the clock\n"
        "                   time belongs. Max 15 words.\n"
        "  caregiver_vague  Short implicit fragment. Must NOT name the specific\n"
        "                   activity or objects. Informal register. Max 7 words.\n"
        "  schedule  Calendar events list:\n"
        '    [{"title":"...","start_time":"{TIME}","location":null,"description":null}]\n'
        "  Rules for schedule:\n"
        "  - title MUST name the specific activity from the concepts\n"
        '    (e.g. "Ski run", "Swimming lesson": not "Activity" or "Event").\n'
        '  - start_time MUST be the literal string "{TIME}".\n'
        "  - Use [] ONLY for CLEARLY passive, home-based activities:\n"
        "    sleeping, resting, watching TV at home, eating at home, bathing.\n"
        "  - For ANY other activity — sport, class, therapy, outing, shopping,\n"
        "    appointment, park, farm, pool, gym, school, concert, ride, dance\n"
        "    — include EXACTLY ONE event. When in doubt, include an event.\n"
        "  - DEFAULT TO ADDING AN EVENT unless unmistakably domestic/passive.\n\n"

        "CRITICAL SEMANTIC CONSTRAINT:\n"
        "time_of_day MUST reflect when this activity naturally happens.\n"
        "  breakfast -> morning  |  dinner -> evening  |  sleep/nap -> night\n"
        "  concert/friday night -> night  |  afternoon tea -> afternoon\n"
        "The schedule title and caregiver_clear MUST match the input activity.\n\n"

        "Examples:\n"
        f"{_EXAMPLES_TEXT}\n\n"

        f'Input: sentence="{sentence}", concepts=[{concepts_str}]\n'
        "Output (compact single-line JSON):"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

## 6. JSON Extraction, Stopping Criteria & Validation

This cell defines all the post-processing logic that runs **after** generation to clean and validate the model's raw output.

- **`_extract_json`**: parses the first valid `{...}` object from the raw text. It specifically searches after the first `Output:` token, because the model tends to generate fake follow-up examples after the real answer (continuation bias), and we always want the first one.
- **`_BatchStopOnSubstring`**: a custom `StoppingCriteria` that halts generation as soon as every sequence in the batch has produced a stop string (`\nInput:` or `\nOutput:`). Without this, the model would keep generating until `MAX_NEW_TOKENS` even after finishing the JSON.
- **`_validate`**: the main sanitisation step. It re-derives `time_of_day` from the sampled `event_time` (so the model cannot influence it), soft-trims both caregiver fields, delegates schedule alignment to `_align_schedule`, and **repairs** `caregiver_clear` when the model omits `{TIME}` (by appending `" at {TIME}"`) instead of discarding the annotation.
- **`_render_time`**: called once at save time to substitute every `{TIME}` placeholder with the concrete `event_time` value in `caregiver_clear` and `schedule[].start_time`. The log always stores `{TIME}`; only the final parquet gets the resolved times.

In [ ]:
def _parse_time(t_str: str) -> tuple[int, int] | None:
    """Parse a HH:MM string into (hour, minute). Returns None on failure."""
    try:
        h, m = map(int, t_str.split(":"))
        return h, m
    except:
        return None


def _hour_to_slot(hour: int) -> str:
    """Convert an hour (0-23) to the corresponding public time_of_day label."""
    if 5  <= hour <= 12: return "morning"
    if 13 <= hour <= 17: return "afternoon"
    if 18 <= hour <= 20: return "evening"
    return "night"


def _fallback(evt_time: str, tod: str) -> dict:
    """Return an empty annotation dict used when validation fails."""
    return {
        "caregiver_clear": "",
        "caregiver_vague": "",
        "time_of_day"    : tod,
        "event_time"     : evt_time,
        "schedule"       : [],
        "tod_selection"  : None,
    }


def _extract_json(text: str) -> dict | None:
    """Extract the first valid JSON object from the model's raw output string.

    We search after the first 'Output:' token rather than the last one,
    because the model sometimes generates fake follow-up examples after
    the real answer (continuation bias). Taking the first occurrence
    ensures we always read the actual response.
    """
    text = re.sub(r"```(?:json)?", "", text).strip()  # strip markdown fences if present

    def _first_balanced_json(s: str) -> dict | None:
        """Find and parse the first balanced {...} block in s."""
        start = s.find("{")
        if start < 0:
            return None
        depth, in_str, esc = 0, False, False
        for i, ch in enumerate(s[start:], start):
            if esc:
                esc = False; continue
            if ch == "\\" and in_str:
                esc = True; continue
            if ch == '"':
                in_str = not in_str; continue
            if in_str:
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(s[start: i + 1])
                    except json.JSONDecodeError:
                        return None
        return None

    # Strategy 1: look for JSON after the first "Output:" marker
    first_out = text.find("Output:")
    if first_out >= 0:
        result = _first_balanced_json(text[first_out:])
        if result is not None:
            return result

    # Strategy 2: just grab the first '{' anywhere in the output
    return _first_balanced_json(text)


class _BatchStopOnSubstring(StoppingCriteria):
    """Stop batched generation once EVERY sequence has produced a stop string.

    Without this, the model would keep generating fake follow-up examples
    after the real JSON output, wasting time and VRAM. Generation stops
    as soon as all sequences in the batch have hit a stop string.
    """

    def __init__(self, stop_token_seqs: list[list[int]]):
        self._seqs = [torch.tensor(s) for s in stop_token_seqs if s]
        self._done: torch.Tensor | None = None

    def __call__(self, input_ids: torch.LongTensor, scores, **kwargs) -> bool:
        bsz = input_ids.shape[0]
        if self._done is None:
            self._done = torch.zeros(bsz, dtype=torch.bool, device=input_ids.device)
        for seq in self._seqs:
            n = len(seq)
            if input_ids.shape[1] >= n:
                tail    = input_ids[:, -n:]
                matches = (tail == seq.to(input_ids.device)).all(dim=1)
                self._done |= matches  # once done, always done
        return bool(self._done.all())  # stop only when every sequence is done


def _trim_words(text: str, max_words: int) -> str:
    """Soft-trim text to max_words, with a 20% tolerance to avoid over-cutting.

    When truncation is needed, we try to cut at the last sentence-ending
    punctuation mark in the first half of the truncated text.
    """
    words = str(text).split()
    if len(words) <= int(max_words * 1.2):
        return str(text)
    truncated = " ".join(words[:max_words])
    for punct in (".", "!", "?"):
        last = truncated.rfind(punct)
        if last > len(truncated) // 2:
            return truncated[: last + 1]
    return truncated


def _align_schedule(raw_schedule, event_time: str, time_of_day: str) -> list[dict]:
    """Validate and align schedule events produced by the model.

    Rules applied:
    - start_time is always forced to the literal "{TIME}" placeholder; the
      concrete clock time lives in the parent event_time field and is
      substituted downstream.  This prevents concrete HH:MM values from
      leaking into the log regardless of what the model emitted.
    - Events with a title shorter than 4 characters are discarded (too generic).
    - Duplicate titles (case-insensitive) are dropped so the model cannot
      produce two identical events for the same row.
    - At most 2 events are kept.
    """
    if not isinstance(raw_schedule, list):
        return []

    aligned     = []
    seen_titles : set[str] = set()

    for event in raw_schedule[:2]:
        if not isinstance(event, dict):
            continue

        title = str(event.get("title", "") or "").strip()
        if len(title) < 4:
            continue  # discard generic or empty titles

        norm = title.lower()
        if norm in seen_titles:
            continue  # skip duplicate titles
        seen_titles.add(norm)

        # Always use the literal {TIME} placeholder regardless of what the model
        # generated.  The actual clock time is stored in the parent event_time
        # field and will be substituted downstream.
        aligned.append({
            "title"      : title,
            "start_time" : "{TIME}",
            "location"   : event.get("location"),
            "description": event.get("description"),
        })

    return aligned


def _validate(raw: dict | None, event_time: str) -> tuple[dict, bool]:
    """Sanitise the model's raw JSON output and return (validated_dict, success).

    Important: time_of_day is always derived from event_time (which was sampled
    deterministically), never taken from the model's output. This ensures the
    final time_of_day is always consistent with the actual clock time.
    """
    parsed_hms  = _parse_time(event_time)
    h           = parsed_hms[0] if parsed_hms else 9
    time_of_day = _hour_to_slot(h)  # derived from event_time, not from the model

    if raw is None or not isinstance(raw, dict):
        return _fallback(event_time, time_of_day), False

    caregiver_clear = _trim_words(str(raw.get("caregiver_clear", "")), 15)
    caregiver_vague = _trim_words(str(raw.get("caregiver_vague", "")), 7)

    # Both caregiver fields must be non-empty.
    if not caregiver_clear or not caregiver_vague:
        return _fallback(event_time, time_of_day), False

    # If the model produced a valid sentence but forgot {TIME} (common for
    # stative or purely descriptive activities such as "a dog wag its tail"),
    # we repair the field rather than rejecting the whole annotation.
    # The repair appends " at {TIME}" so that _render_time can substitute
    # the concrete clock time in the final output just like any other row.
    if "{TIME}" not in caregiver_clear:
        caregiver_clear = caregiver_clear.rstrip(".!?,") + " at {TIME}."

    schedule = _align_schedule(raw.get("schedule", []), event_time, time_of_day)

    return {
        "caregiver_clear": caregiver_clear,
        "caregiver_vague": caregiver_vague,
        "time_of_day"    : time_of_day,
        "event_time"     : event_time,
        "schedule"       : schedule,
    }, True


def _render_time(annotation: dict) -> dict:
    """Replace every {TIME} placeholder with the concrete event_time value.

    This is called once, at the final output stage, on every row in the
    annotated DataFrame. The log always stores {TIME} (for resumability and
    back-compat); the substitution only happens in the parquet output.

    Fields touched:
    - caregiver_clear: the literal string {TIME} → HH:MM
    - schedule[].start_time: the literal string {TIME} → HH:MM
    """
    evt = str(annotation.get("event_time", ""))
    if not evt:
        return annotation

    out = dict(annotation)
    out["caregiver_clear"] = str(out.get("caregiver_clear", "")).replace("{TIME}", evt)

    raw_sched = out.get("schedule") or []
    rendered_sched = []
    for ev in raw_sched:
        ev2 = dict(ev)
        if ev2.get("start_time") == "{TIME}":
            ev2["start_time"] = evt
        rendered_sched.append(ev2)
    out["schedule"] = rendered_sched

    return out


print("Cell 6 loaded: extraction, stopping criteria, validation and time rendering defined.")

## 7. Results Helpers

Two small utilities used at the end of each annotation pass:
- **`apply_results`**: writes the accumulated valid annotations back into a copy of the original DataFrame.
- **`assign_split`**: tags each row based on which caregiver fields are present (`both` / `clear` / `vague` / `none`), useful for downstream dataset splitting.

In [ ]:
def apply_results(orig_df: pd.DataFrame, results: dict[int, dict]) -> pd.DataFrame:
    """Write annotation results into a copy of the original DataFrame."""
    df = orig_df.copy()
    # Ensure all output columns exist before writing
    for col in ("caregiver_clear", "caregiver_vague", "time_of_day", "event_time", "schedule", "tod_selection"):
        if col not in df.columns:
            df[col] = None
    for idx, fields in results.items():
        if idx in df.index:
            for col, val in fields.items():
                df.at[idx, col] = val
    return df


def assign_split(row: pd.Series) -> str:
    """Classify a row based on which caregiver fields are populated."""
    has_clear = bool(str(row.get("caregiver_clear", "")).strip())
    has_vague = bool(str(row.get("caregiver_vague", "")).strip())
    if has_clear and has_vague:
        return "both"
    return "clear" if has_clear else ("vague" if has_vague else "none")


print("Helpers defined.")

## 8. Annotation Log Helpers

The log file (`annotation_log.jsonl`) serves two purposes:
1. **Resumability**: on restart, already-annotated rows are loaded from the log and skipped, so we never re-generate rows that already succeeded.
2. **Auditability**: each entry stores the original sentence, concepts, raw model output and parsed result, making it easy to inspect failures.

Only rows with a non-empty `caregiver_clear` are considered valid and added to `_logged_idxs`.

In [ ]:
# Global set of row indices that have already been successfully annotated.
# Populated by _init_logged_idxs() at the start of each annotation pass.
_logged_idxs: set[int] = set()


def _init_logged_idxs() -> dict[int, dict]:
    """Read the log file and return a dict of already-valid annotations.

    Also populates the global _logged_idxs set so that annotate_dataset
    knows which rows to skip.

    Back-compat sanitisation applied to every loaded entry:
    - schedule events whose start_time is a concrete HH:MM (written by the
      old notebook) are rewritten to the "{TIME}" placeholder so that all
      entries in the returned dict are consistent with the new contract.
    - entries whose caregiver_clear does not contain "{TIME}" are treated as
      invalid and excluded; they will be re-processed in the next pass.
    """
    global _logged_idxs
    valid_data  = {}
    n_sanitised = 0
    n_skipped   = 0

    if not LOG_PATH.exists():
        return valid_data

    with open(LOG_PATH, "r", encoding="utf-8") as fh:
        for line in fh:
            try:
                entry  = json.loads(line)
                idx    = int(entry.get("idx", -1))
                parsed = entry["parsed"]

                caregiver_clear = parsed.get("caregiver_clear", "")

                # Entries without the {TIME} placeholder in caregiver_clear are
                # invalid under the new contract and must be re-generated.
                if not caregiver_clear or "{TIME}" not in caregiver_clear:
                    n_skipped += 1
                    continue

                # Normalise schedule: replace any concrete start_time with {TIME}.
                for ev in parsed.get("schedule", []):
                    if isinstance(ev, dict) and ev.get("start_time") != "{TIME}":
                        ev["start_time"] = "{TIME}"
                        n_sanitised += 1

                _logged_idxs.add(idx)
                valid_data[idx] = parsed

            except:
                continue  # skip malformed lines silently

    log.info(
        "Log loaded: %d valid annotations found "
        "(%d schedule entries sanitised, %d entries skipped for re-generation).",
        len(_logged_idxs), n_sanitised, n_skipped,
    )
    return valid_data


def log_annotation(
    orig_idx  : int,
    sentence  : str,
    concepts  : list,
    raw_output: str,
    parsed    : dict,
) -> None:
    """Append a successful annotation to the JSONL log file.

    We skip rows that are already in _logged_idxs to avoid duplicates
    in case the function is called multiple times for the same index.
    """
    global _logged_idxs
    if orig_idx in _logged_idxs:
        return

    entry = {
        "ts"        : datetime.utcnow().isoformat(),
        "idx"       : orig_idx,
        "sentence"  : sentence,
        "concepts"  : concepts,
        "raw_output": raw_output,
        "parsed"    : parsed,
    }
    with open(LOG_PATH, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(entry, ensure_ascii=False) + "\n")
    _logged_idxs.add(orig_idx)

## 9. Run Full Annotation

### Annotation flow

For each batch of rows the pipeline follows these steps:

1. **Semantic pre-check**: before calling the model, we run `_semantic_check` on each sentence. If a strong keyword is found (e.g. "sleep": night), we store a forced `time_of_day` that will override the model's choice.

2. **Batched generation**: all prompts in the batch are tokenised together and passed to `model.generate()` in a single forward pass. The custom `_BatchStopOnSubstring` stopping criterion halts generation early once every sequence has produced a stop token.

3. **Post-processing per row**:
   - Extract the JSON from the raw output with `_extract_json`
   - Determine the final `time_of_day`: use the forced value if available, otherwise trust the model; fall back to a weighted random draw if the model produced an invalid label
   - Sample a concrete `event_time` within the correct slot with `_sample_event_time`
   - Validate and sanitise everything with `_validate`

4. **Retry**: rows that fail validation stay in `pending` and are retried up to `MAX_ROW_RETRIES` times within the same batch.

### Outer retry loop

After each full pass, we count rows that still have an empty `caregiver_clear`. If any remain, we re-run `annotate_dataset` up to `MAX_ANNOTATION_RETRIES` times. Because the log is updated after every successful row, each pass only processes what is still missing.

In [ ]:
# Valid public time_of_day labels the model is expected to produce
VALID_TIME_OF_DAY = {"morning", "afternoon", "evening", "night"}


def annotate_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Run one full annotation pass over df, skipping already-logged rows."""

    # Load existing valid annotations and update _logged_idxs
    current_valid_results = _init_logged_idxs()

    todo = [(idx, row) for idx, row in df.iterrows() if idx not in _logged_idxs]
    if not todo:
        log.info("All rows already annotated: nothing to do.")
        return apply_results(df, current_valid_results)

    batches = [todo[i : i + BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
    log.info("Rows to process: %d | Batches: %d", len(todo), len(batches))

    # Pre-tokenise the stop strings once; used by _BatchStopOnSubstring
    _stop_token_seqs = [
        tokenizer.encode(s, add_special_tokens=False)
        for s in ["\nInput:", "\nOutput:"]
    ]

    for _, batch in enumerate(batches, 1):
        idxs = [idx for idx, _ in batch]
        rows = [row for _, row in batch]

        # Step 1: semantic pre-check (before generation)
        # forced_tods[i] is non-None when the sentence contains a strong keyword
        forced_tods = [
            _semantic_check(r["sentence"], r.get("concept", []))
            for r in rows
        ]

        pending = list(range(len(rows)))  # indices of rows still needing a valid annotation

        for attempt in range(1, MAX_ROW_RETRIES + 1):
            if not pending:
                break

            # Build and tokenise prompts only for the rows still pending
            prompts = [
                _build_prompt(rows[pi]["sentence"], rows[pi].get("concept", []))
                for pi in pending
            ]
            enc = tokenizer(
                prompts,
                return_tensors  = "pt",
                padding         = True,
                truncation      = True,
                max_length      = MAX_PROMPT_LENGTH,
            ).to(model.device)

            # Step 2: batched generation
            with torch.no_grad():
                out_ids = model.generate(
                    **enc,
                    max_new_tokens    = MAX_NEW_TOKENS,
                    do_sample         = True,
                    temperature       = 0.7,
                    stopping_criteria = StoppingCriteriaList(
                        [_BatchStopOnSubstring(_stop_token_seqs)]
                    ),
                )

            # Decode only the newly generated tokens (strip the prompt)
            decoded = tokenizer.batch_decode(
                out_ids[:, enc["input_ids"].shape[1]:],
                skip_special_tokens=True,
            )

            # Step 3: post-process each decoded output
            still_pending = []
            for i, pi in enumerate(pending):
                raw_json  = _extract_json(decoded[i])

                # Determine the final time_of_day:
                #   a) forced by keyword match (highest priority)
                #   b) chosen by the model (if valid)
                #   c) random weighted fallback (if the model produced garbage)
                forced    = forced_tods[pi]
                model_tod = raw_json.get("time_of_day") if isinstance(raw_json, dict) else None
                final_tod = forced if forced is not None else model_tod
                if final_tod not in VALID_TIME_OF_DAY:
                    # Use a fixed ordered list so weights are deterministic:
                    # morning 30%, afternoon 30%, evening 20%, night 20%
                    final_tod = _random.choices(
                        ["morning", "afternoon", "evening", "night"],
                        weights=[0.30, 0.30, 0.20, 0.20],
                    )[0]

                # Determine how time_of_day was selected (for audit attribute)
                if forced is not None:
                    tod_selection = "keyword-forced"
                elif model_tod in VALID_TIME_OF_DAY:
                    tod_selection = "model-forced"
                else:
                    tod_selection = "sampled"

                # Sample a concrete clock time within the determined slot
                _, _, evt = _sample_event_time(final_tod)

                validated, ok = _validate(raw_json, evt)

                if ok:
                    # Stamp the selection method onto the validated annotation
                    validated["tod_selection"] = tod_selection
                    log_annotation(
                        int(idxs[pi]),
                        rows[pi]["sentence"],
                        list(rows[pi].get("concept", [])),
                        decoded[i],
                        validated,
                    )
                    current_valid_results[idxs[pi]] = validated
                else:
                    still_pending.append(pi)  # will be retried in the next attempt

            pending = still_pending
            del enc, out_ids
            torch.cuda.empty_cache()  # free VRAM between retry attempts

        # Rows still pending after all retries will be picked up in the outer loop
        for pi in pending:
            log.warning(
                "idx=%d failed after %d retries: will be retried in the next outer pass.",
                idxs[pi], MAX_ROW_RETRIES,
            )

    annotated          = apply_results(df, current_valid_results)
    annotated["split"] = annotated.apply(assign_split, axis=1)
    return annotated


# ── Outer retry loop ──────────────────────────────────────────────────────────
# We run multiple full passes to recover rows that failed in previous passes.
# Each pass only processes rows not yet in the log, so it gets cheaper over time.
t_start = time.time()
for _attempt in range(1, MAX_ANNOTATION_RETRIES + 2):
    log.info("=== Annotation pass %d ===", _attempt)
    df_annotated = annotate_dataset(df_raw)

    n_failed = (
        df_annotated["caregiver_clear"].isna().sum()
        + (df_annotated["caregiver_clear"] == "").sum()
    )

    if n_failed == 0:
        log.info("Done! All rows successfully annotated.")
        break
    log.warning("%d rows still missing: starting recovery pass.", n_failed)

# ── Render {TIME} placeholders before saving ─────────────────────────────────
# The log and in-memory dicts keep {TIME} as a literal string (for resumability).
# Here we substitute every {TIME} with the concrete event_time for the final output.
def _apply_render_time(row: pd.Series) -> pd.Series:
    """Apply _render_time to a single DataFrame row."""
    ann = {
        "event_time"     : row.get("event_time", ""),
        "caregiver_clear": row.get("caregiver_clear", ""),
        "schedule"       : row.get("schedule", []),
    }
    rendered = _render_time(ann)
    row = row.copy()
    row["caregiver_clear"] = rendered["caregiver_clear"]
    row["schedule"]        = rendered["schedule"]
    return row

df_annotated = df_annotated.apply(_apply_render_time, axis=1)

# Save the final annotated DataFrame
df_annotated.to_parquet(ANNOTATED_PATH, index=False)
log.info("Pipeline finished in %.0f s: output saved to %s", time.time() - t_start, ANNOTATED_PATH)